# Technologies.csv Generation — Projection 2035 — Norte Amazónica

Generates one `Technologies.csv` per cluster (C1–C5) for EnergyScope ESMC, **2035 projection**,
copied from `analyse data ramp/reality/technologies.ipynb` (2025 reality scenario) with four
mandatory corrections for the projection horizon plus a new horizon-specific existing-fleet rule.

Base: sufficiency output (`../sufficiency/output_energyscope/C{k}/Technologies.csv`) — the same 2025
base used by the reality scenario, since no sufficiency scenario exists for 2035/2050 yet. Structural
rows (LED lighting, stove capacity factor, storage technology list, etc.) are assumed unchanged.
Only values documented below are overwritten; all other rows are inherited unchanged from sufficiency.

**Cost/lifetime block:** `c_inv`, `c_maint`, `gwp_constr`, `lifetime` (plus `Category`, `Subcategory`,
`Technologies name`, `Comment`) are **not** produced by the sufficiency base above — that file only
carries the 8 capacity columns (`c_p`, `fmin_perc`, `fmax_perc`, `f_min`, `f_max`, `f_min_prod`,
`f_max_prod`). Those 8 columns are merged in (rule (f) below) from the deployed, calibrated 2025
reality catalog — see the note on the previous build below.

**Rules applied:**
- **(a) Electricity generators** — `f_max = f_min` (locks capacity to existing installed amounts; no
  new construction), **excluding `PV_HS`/`HS_DIESEL`** (see note below).
- **(b) Off-grid SHS kits** — `PV_HS` / `HS_DIESEL`: `f_min = 0` (the 2012-vintage legacy kits are past
  their catalogue lifetime — PV_HS 20y, HS_DIESEL 5y — by this horizon), but `f_max` is left
  **unconstrained** (inherited from sufficiency, `1e15`) so the model can build new SHS capacity.
  This differs from the reality/2025 notebook, which also zeroes `f_max` (no new off-grid
  construction in a status-quo scenario) — correction 1 below.
- **(c) Storage** — `f_max = f_min` for all battery/storage technologies **except `BATT_HS`**
  (correction 2 below), which is left untouched by both rule (b) and (c) and keeps the sufficiency
  base's own values (`f_min = 0`, `f_max = 1e15`, also past its 10y catalogue lifetime).
- **(d) Stoves** — `f_min` recalculated from Census 2024 cooking fuel data (**not** projected to this
  horizon — see "Missing entries" at the end of this notebook); `f_max` unchanged from sufficiency.
- **(e) Existing generation fleet (new, horizon-specific)** — `GENSET_DIESEL` / `PV_UTILITY` `f_min`
  set explicitly for 2035 (Section 1bis), then locked by rule (a) as usual.
- **(f) Cost/lifetime block (new)** — `c_inv`, `c_maint`, `gwp_constr`, `lifetime` (plus the
  `Category`/`Subcategory`/`Technologies name`/`Comment` structural columns) are merged in verbatim
  from `Data/2025/reality/02_REF_REGION/Technologies.csv`, the deployed and calibrated Norte Amazónica
  2025 catalog. A blocking assertion (Section 5) checks these four columns are byte-identical to that
  reference for every one of the 272 technologies, with **no exceptions** — cost trajectories for this
  horizon are not modelled yet, so nothing should differ. The notebook halts before writing any file
  if the assertion fails.

**Corrections applied vs. `analyse data ramp/reality/technologies.ipynb`:**
1. Off-grid rule (b): the `f_max = 0.0` override for `PV_HS`/`HS_DIESEL` is removed — legacy kits are
   dead (`f_min = 0`) but the model must be able to build new kits.
2. `BATT_HS` removed from `STORAGE_TECHS` (rule c) — otherwise `f_max = f_min = 0` and no home battery
   is buildable, while the PV_HS/BATT_HS coupling constraint is active by default (`.mod:151 = 1`).
3. `Layers_in_out.csv` reference-equality assertion added (Section 1, same check already used in the
   `demande.ipynb` notebooks) — this was the only unprotected reader of that file, and it feeds a file
   that enters an EnergyScope run.
4. `OUT_DIR` suffixed by year (`output_energyscope_2035`) — no projection output can land in a 2025 path.

**Consequence of correction 1, discovered while implementing it:** the shared `Layers_in_out.csv` now
flags `PV_HS`/`HS_DIESEL` as `ELECTRICITY` generators (54 generators identified here, vs. 52 when
`reality/technologies.ipynb` last ran) — they weren't classified that way when the reality notebook
was last executed. Left as-is, rule (a) would therefore re-lock `f_max = f_min` for `PV_HS`/`HS_DIESEL`
*before* rule (b) even runs, silently undoing correction 1. Rule (a)'s mask excludes `OFF_GRID_TECHS`
to prevent this (Section 4). This is a necessary consequence of the current data, not a fifth
independent correction; `reality/technologies.ipynb` itself was left untouched.

**Bug found and fixed in this build — rule (f), cost/lifetime block:** the previous build of this
notebook (and of `2050/technologies.ipynb`) never merged a cost/lifetime block in at all — the CSV
that was actually copied into `Data/2035/*/02_REF_REGION/Technologies.csv` was assembled by hand
outside of any notebook, starting from the generic pre-Bolivia EnergyScope template (same lineage as
`Data/2021/02_REF_REGION/Technologies.csv` and `Data/2030/...`) instead of from the calibrated 2025
reality catalog. That silently reverted `GRID.c_inv` (0 → 810.19), `HVAC_LINE.c_inv`/`c_maint`
(2/0.04 → 1532/30.64), `PV_HS.c_inv`/`c_maint`/`lifetime` (2730/10/20 → 4226.13/7.78/30), and
`HS_DIESEL.lifetime` (5 → 30), accounting for ~96% of the resulting cost discrepancy. Rule (f) above
and the Section 5 assertion are the fix — sourcing this block from a notebook step, with the reference
file path recorded, rather than an untracked manual edit.

In [1]:
import os
import pandas as pd

OUT_DIR  = "output_energyscope_2035"
SUFF_DIR = "../sufficiency/output_energyscope"

# Rule (f): cost/lifetime block source — the deployed, calibrated Norte Amazónica 2025 catalog.
# The sufficiency base above only carries capacity columns; c_inv/c_maint/gwp_constr/lifetime are
# merged in from here (Section 5), not projected in any way for this horizon.
COST_REF_PATH = "../../../EnergyScope_BO_nord_amazonia/Data/2025/reality/02_REF_REGION/Technologies.csv"

# NREL ATB 2024, Moderate/Mid cost trajectory. Dimensionless ratio applied to the 2025
# reality catalog's c_inv (never an absolute substitution) -- immune to the catalog's mixed
# cost-year vintages (TS_DEC in 2015 EUR, PV_HS/BATT_HS in 2024 EUR, DEC_SOLAR undated).
NREL_ATB_C_INV_RATIO = {
    "PV_UTILITY": {2035: 0.600, 2050: 0.458},
    "BATT_LI":    {2035: 0.757, 2050: 0.555},
    "PV_HS":      {2035: 0.65,  2050: 0.46},
    "BATT_HS":    {2035: 0.76,  2050: 0.61},
}

CLUSTERS = {
    1: ["Exaltación", "Reyes", "Santa_Rosa_Beni", "Ixiamas"],
    2: ["Bolpebra"],
    3: ["Guayaramerín", "Riberalta", "Puerto_Gonzalo_Moreno"],
    4: ["Bella_Flor", "Filadelfia", "Ingavi", "Nueva_Esperanza", "Porvenir",
        "Puerto_Rico", "San_Lorenzo", "San_Pedro", "Santa_Rosa_Pando",
        "Santos_Mercado", "Sena", "Villa_Nueva"],
    5: ["Cobija"],
}

# Rule (e): existing generation fleet carried over from 2025 (reality scenario), still within its
# expected service life at 2035. PV_UTILITY C4 adds El Sena (400 kWp solar, EGSA/ENDE Guaracachi,
# commissioned end-2018) — absent from the 2025 dataset. Applied before Rule (a) locks f_max = f_min.
EXISTING_FLEET_FMIN_GW = {
    "GENSET_DIESEL": {3: 0.05333, 4: 0.00742, 5: 0.02736},
    "PV_UTILITY":    {4: 0.00040, 5: 0.00510},
}

# Rule (g): DEC_SOLAR.f_max — the sufficiency base above is stale on this column (local
# ../sufficiency/output_energyscope/C{k}/Technologies.csv carries f_max=0, but the actually
# deployed Data/2025/sufficiency/C{k}/Technologies.csv has f_max=1e15 for every cluster).
# Separately, inheriting from Data/2025/reality would also be wrong here: reality/reality_phase2
# deliberately lock DEC_SOLAR (Source A households do not own a solar water heater today), but
# Sources B and C get the sufficiency bundle in every projection scenario, and sufficiency (like
# reality_access) leaves DEC_SOLAR investment open. Inherit explicitly, per cluster, from the
# deployed reality_access reference instead of trusting either the stale local base or reality.
DEC_SOLAR_FMAX_REF_PATH_TMPL = "../../../EnergyScope_BO_nord_amazonia/Data/2025/reality_access/C{k}/Technologies.csv"
DEC_SOLAR_FMAX_BY_CLUSTER = {}
for k in range(1, 6):
    ra = pd.read_csv(DEC_SOLAR_FMAX_REF_PATH_TMPL.format(k=k), sep=";", index_col=0)
    ra.index = ra.index.str.strip()
    DEC_SOLAR_FMAX_BY_CLUSTER[k] = float(ra.loc["DEC_SOLAR", "f_max"])
print("DEC_SOLAR f_max by cluster, from the deployed 2025 reality_access reference:", DEC_SOLAR_FMAX_BY_CLUSTER)

# Rule (h): PV_HS / HS_DIESEL f_max_prod -- absolute annual production ceiling [GWh/y],
# independent of f_max (capacity). Left at the sufficiency base's inactive default (1e15) by
# every rule above: ESMC_model_AMPL.mod only generates the f_max_prod_abs constraint when
# f_max_prod[c,j] < 1e14 (restricted indexing, mod:871), so at the default the cap is not just
# large -- the constraint row does not exist at all. Home-system techs are also excluded from
# the network-loss assiette (TECH_HS diff, mod:632-633), making their kWh cheaper than a
# grid-connected genset, so an unconstrained optimizer over-builds them into the grid-connected
# ELECTRICITY balance -- not a share_dispersion=0 effect, this cap is a separate mechanism.
# Cap each of PV_HS and HS_DIESEL (independently -- the .mod indexes production per technology,
# not their sum, a known formulation limit) at the cluster's dispersed demand. f_min_prod stays
# at its default 0 -- do NOT pin f_min_prod = f_max_prod, that breaks the solver.
DISPERSED_DEMAND_REF_PATH = "../../analyse_GIS_phase2_projections/output/2035/cluster_summary.csv"
_cluster_summary = pd.read_csv(DISPERSED_DEMAND_REF_PATH, index_col="Cluster")
DISPERSED_DEMAND_GWH_BY_CLUSTER = {
    k: float(_cluster_summary.loc[f"C{k}", "demande_dispersee_GWh"]) for k in range(1, 6)
}
print("Dispersed demand by cluster [GWh/y], from cluster_summary.csv:", DISPERSED_DEMAND_GWH_BY_CLUSTER)

# Load sufficiency output as base — only modified rows are overwritten
suff = {}
for k in range(1, 6):
    path = os.path.join(SUFF_DIR, f"C{k}", "Technologies.csv")
    df = pd.read_csv(path, sep=";")
    df["Technologies param"] = df["Technologies param"].str.strip()
    suff[k] = df
print("Loaded sufficiency base for C1–C5")

DEC_SOLAR f_max by cluster, from the deployed 2025 reality_access reference: {1: 1000000000000000.0, 2: 1000000000000000.0, 3: 1000000000000000.0, 4: 1000000000000000.0, 5: 1000000000000000.0}
Dispersed demand by cluster [GWh/y], from cluster_summary.csv: {1: 2.0166, 2: 0.1301, 3: 0.7095, 4: 1.5231, 5: 0.0}
Loaded sufficiency base for C1–C5


## 1. Electricity generation — rule (a)

`f_max = f_min` for all technologies with positive `ELECTRICITY` output in `Layers_in_out.csv`.
This locks each cluster to its existing installed capacity (from AETN 2024 / Census data already encoded in the sufficiency base).
If `f_min = 0`, then `f_max = 0` — no new construction of any generator type.

In [2]:
# Source: Layers_in_out.csv — techs with positive ELECTRICITY coefficient are generators
lio = pd.read_csv("../data/Layers_in_out.csv", sep=";")
lio.columns = [c.strip() for c in lio.columns]
tech_col = lio.columns[0]

# Correction 3: Layers_in_out.csv reference-equality check (same pattern already used in the
# demande.ipynb notebooks). This reader was previously unprotected, and the file it produces feeds
# directly into Technologies.csv, which enters an EnergyScope run. Efficiencies are frozen at 2025
# values for both projection horizons, so the reference is the 2025 reality file.
LIO_REFERENCE_PATH = "../../../EnergyScope_BO_nord_amazonia/Data/2025/reality/00_INDEP/Layers_in_out.csv"
lio_check_local = pd.read_csv("../data/Layers_in_out.csv", sep=";", header=0, index_col=0)
lio_check_reference = pd.read_csv(LIO_REFERENCE_PATH, sep=";", header=0, index_col=0)

problems = []
only_local_rows = sorted(set(lio_check_local.index) - set(lio_check_reference.index))
only_reference_rows = sorted(set(lio_check_reference.index) - set(lio_check_local.index))
if only_local_rows:
    problems.append(f"technologies only in local file: {only_local_rows}")
if only_reference_rows:
    problems.append(f"technologies only in EnergyScope reference: {only_reference_rows}")

only_local_cols = sorted(set(lio_check_local.columns) - set(lio_check_reference.columns))
only_reference_cols = sorted(set(lio_check_reference.columns) - set(lio_check_local.columns))
if only_local_cols:
    problems.append(f"layers only in local file: {only_local_cols}")
if only_reference_cols:
    problems.append(f"layers only in EnergyScope reference: {only_reference_cols}")

common_rows = sorted(set(lio_check_local.index) & set(lio_check_reference.index))
common_cols = sorted(set(lio_check_local.columns) & set(lio_check_reference.columns))
changed_rows = sorted(
    tech for tech in common_rows
    if not lio_check_local.loc[tech, common_cols].equals(lio_check_reference.loc[tech, common_cols])
)
if changed_rows:
    problems.append(f"technologies with different coefficients: {changed_rows}")

if problems:
    raise ValueError(
        f"Layers_in_out.csv differs from the EnergyScope reference ({LIO_REFERENCE_PATH}): "
        + "; ".join(problems)
    )
print(f"OK — Layers_in_out.csv matches the EnergyScope reference ({LIO_REFERENCE_PATH})")

ELECTRICITY_GENERATORS = set(
    lio.loc[(lio["ELECTRICITY"] > 0) & (lio[tech_col] != "ELECTRICITY"), tech_col].tolist()
)
print(f"Identified {len(ELECTRICITY_GENERATORS)} electricity generators:")
print(sorted(ELECTRICITY_GENERATORS))

OK — Layers_in_out.csv matches the EnergyScope reference (../../../EnergyScope_BO_nord_amazonia/Data/2025/reality/00_INDEP/Layers_in_out.csv)
Identified 54 electricity generators:
['BFB_ST_BIOMASS', 'BIOMASS_TO_DIESEL', 'BIOMASS_TO_GASOLINE', 'BIOMASS_TO_JET_FUEL', 'BIOMASS_TO_LFO', 'BIOMASS_TO_METHANOL', 'BIOMASS_TO_POWER', 'BIOWASTE_TO_DIESEL', 'BIOWASTE_TO_GASOLINE', 'BIOWASTE_TO_JET_FUEL', 'BIOWASTE_TO_LFO', 'BIOWASTE_TO_METHANOL', 'BIO_HYDROLYSIS', 'CCGT', 'CCGT_AMMONIA', 'CCGT_SUR', 'CFB_ST_BIOMASS', 'COAL_IGCC', 'COAL_US', 'DEC_ADVCOGEN_GAS', 'DEC_ADVCOGEN_H2', 'DEC_COGEN_GAS', 'DEC_COGEN_OIL', 'DHN_COGEN_GAS', 'DHN_COGEN_WASTE', 'DHN_COGEN_WOOD', 'ETHANOL_TO_FUELS', 'FB_ST_BIOMASS', 'FUEL_CELL', 'GENSET_DIESEL', 'GEOTHERMAL', 'HS_DIESEL', 'HYDRO_DAM', 'HYDRO_RIVER', 'IND_COGEN_GAS', 'IND_COGEN_WASTE', 'IND_COGEN_WOOD', 'NUCLEAR', 'NUCLEAR_SMR', 'OCGT', 'PT_POWER_BLOCK', 'PV_HS', 'PV_ROOFTOP', 'PV_UTILITY', 'PYROLYSIS_TO_FUELS', 'PYROLYSIS_TO_LFO', 'ST_BIOMASS', 'ST_POWER_BLOCK'

## 2. Off-grid and storage — rules (b) & (c)

**Rule (b):** `PV_HS` and `HS_DIESEL` → `f_min = 0` only (correction 1: the `f_max = 0.0` override
used in the reality/2025 notebook is removed). The 2012-vintage legacy kits are past their catalogue
lifetime (PV_HS 20y, HS_DIESEL 5y) by this horizon, so `f_min = 0`, but `f_max` is left unconstrained
(inherited from the sufficiency base, `1e15`) so the model can invest in new SHS capacity.

**Rule (c):** All other battery/storage technologies → `f_max = f_min` (no new investment). `BATT_HS`
is deliberately excluded from `STORAGE_TECHS` (correction 2) — left untouched, it keeps the
sufficiency base's own values (`f_min = 0`, `f_max = 1e15`; its catalogue lifetime is 10y and it is
likewise past end-of-life). Excluding it matters because otherwise rule (c) would lock it to
`f_max = f_min = 0`, leaving no buildable battery for the PV_HS/BATT_HS coupling constraint
(`.mod` default `= 1`) to pair against new `PV_HS` capacity.

In [3]:
# Rule (b): off-grid techs disabled (f_min only — correction 1: f_max left unconstrained)
OFF_GRID_TECHS = ["PV_HS", "HS_DIESEL"]

# Rule (c): storage techs locked (f_max = f_min)
# Includes electrical, thermal, chemical and vehicle storage.
# BATT_HS is deliberately excluded (correction 2) — see rule (b)/(c) markdown above.
STORAGE_TECHS = [
    # Electrical / electrochemical
    "BATT_LI", "CAES", "BEV_BATT", "PHEV_BATT", "DAM_STORAGE", "PHS",
    # Thermal
    "TS_DEC_DIRECT_ELEC", "TS_DEC_HP_ELEC", "TS_DEC_THHP_GAS",
    "TS_DEC_COGEN_GAS",   "TS_DEC_COGEN_OIL", "TS_DEC_ADVCOGEN_GAS",
    "TS_DEC_ADVCOGEN_H2", "TS_DEC_BOILER_GAS", "TS_DEC_BOILER_WOOD",
    "TS_DEC_BOILER_OIL",  "TS_DHN_DAILY", "TS_DHN_SEASONAL", "TS_HIGH_TEMP", "TS_COLD",
    # Chemical / other
    "GAS_STORAGE", "H2_STORAGE", "CO2_STORAGE", "AMMONIA_STORAGE",
    "METHANOL_STORAGE", "PT_STORAGE", "ST_STORAGE",
]
print("OFF_GRID_TECHS:", OFF_GRID_TECHS)
print(f"STORAGE_TECHS ({len(STORAGE_TECHS)} entries) defined.")

OFF_GRID_TECHS: ['PV_HS', 'HS_DIESEL']
STORAGE_TECHS (27 entries) defined.


## 3. Cooking stoves — rule (d)

`f_min` for `STOVE_WOOD` and `STOVE_LPG` is derived from Census 2024 cooking fuel data
(file: `CSV_final_in_excel.xlsx`, sheet `data`, rows start at row 4):

| Column (0-indexed) | Field |
|---|---|
| 47 | Leña (wood households) |
| 50 | Gas domiciliario por cañería |
| 51 | Gas en garrafa |

$$\text{wood\_hh} = \text{col}_{47}, \quad \text{lpg\_hh} = \text{col}_{50} + \text{col}_{51}$$

$$f_{\min}^{\text{WOOD}} = \frac{\text{wood\_hh} \times 0.001344023}{0.1875 \times 8760} \;[\text{GW}]$$

$$f_{\min}^{\text{LPG}} = \frac{\text{lpg\_hh} \times 0.001344023}{0.1875 \times 8760} \;[\text{GW}]$$

Where $0.001344023$ GWh/hh/yr is the Census-based useful cooking energy intensity,
and $0.1875$ is the stove capacity factor (`c_p`).

**Disambiguation:** Two municipalities are named *Santa Rosa*.
Department Beni → `Santa_Rosa_Beni` → C1; Department Pando → `Santa_Rosa_Pando` → C4.

In [4]:
# Source: Bolivia Census 2024 — CSV_final_in_excel.xlsx
xl = pd.ExcelFile("../../exctraction of data/output/CSV_final_in_excel.xlsx")
raw = xl.parse(xl.sheet_names[0], header=None)
data = raw.iloc[3:].reset_index(drop=True)  # skip 3 header rows

COOKING_HH = {}  # muni_key → (wood_hh, lpg_hh)
for _, row in data.iterrows():
    muni = str(row[3]).strip() if pd.notna(row[3]) else ""
    dept = str(row[1]).strip() if pd.notna(row[1]) else ""
    if not muni or muni == "nan":
        continue
    wood_hh = int(row[47]) if pd.notna(row[47]) else 0
    gas_dom  = int(row[50]) if pd.notna(row[50]) else 0
    gas_gar  = int(row[51]) if pd.notna(row[51]) else 0
    lpg_hh   = gas_dom + gas_gar
    # Disambiguate the two "Santa Rosa" entries
    if muni == "Santa Rosa" and dept == "Beni":
        key = "Santa_Rosa_Beni"
    elif muni == "Santa Rosa" and dept == "Pando":
        key = "Santa_Rosa_Pando"
    else:
        key = muni.replace(" ", "_")
    COOKING_HH[key] = (wood_hh, lpg_hh)

COOK_INTENSITY = 0.001344023  # GWh per household per year (Census 2024 cooking energy intensity)
CP_STOVE       = 0.1875       # capacity factor for all stoves

stove_wood_fmin = {}
stove_lpg_fmin  = {}
for k, munis in CLUSTERS.items():
    wood_total = sum(COOKING_HH.get(m, (0, 0))[0] for m in munis)
    lpg_total  = sum(COOKING_HH.get(m, (0, 0))[1] for m in munis)
    stove_wood_fmin[k] = (wood_total * COOK_INTENSITY) / (CP_STOVE * 8760)
    stove_lpg_fmin[k]  = (lpg_total  * COOK_INTENSITY) / (CP_STOVE * 8760)

print(f"{'':8} {'wood_hh':>9} {'lpg_hh':>9} {'STOVE_WOOD f_min (GW)':>22} {'STOVE_LPG f_min (GW)':>22}")
for k, munis in CLUSTERS.items():
    wood_total = sum(COOKING_HH.get(m, (0, 0))[0] for m in munis)
    lpg_total  = sum(COOKING_HH.get(m, (0, 0))[1] for m in munis)
    print(f"C{k}       {wood_total:>9} {lpg_total:>9} {stove_wood_fmin[k]:>22.7f} {stove_lpg_fmin[k]:>22.7f}")

           wood_hh    lpg_hh  STOVE_WOOD f_min (GW)   STOVE_LPG f_min (GW)
C1            5017      5656              0.0041053              0.0046282
C2             340       452              0.0002782              0.0003699
C3            7014     32377              0.0057394              0.0264934
C4            5775     10520              0.0047256              0.0086083
C5             300     14696              0.0002455              0.0120254


## 4. Assemble capacity overrides (rules a–e)

Builds the capacity-only dataframe per cluster (8 columns, same shape as the sufficiency base).
The cost/lifetime block is merged in separately in Section 5 (rule f) — not saved yet.

In [5]:
capacity_only = {}
for k in range(1, 6):
    df = suff[k].copy()

    # Rule (e): horizon-specific existing generation fleet — override before Rule (a) locks
    # f_max = f_min (see EXISTING_FLEET_FMIN_GW above)
    for tech, by_cluster in EXISTING_FLEET_FMIN_GW.items():
        if k in by_cluster:
            df.loc[df["Technologies param"] == tech, "f_min"] = by_cluster[k]

    # Rule (g): DEC_SOLAR.f_max inherited from the deployed reality_access reference, not the
    # stale sufficiency base (see DEC_SOLAR_FMAX_BY_CLUSTER above) -- not touched by any other
    # rule below (not a generator, not off-grid, not storage), so this is a plain override.
    df.loc[df["Technologies param"] == "DEC_SOLAR", "f_max"] = DEC_SOLAR_FMAX_BY_CLUSTER[k]

    # Rule (a): Lock all electricity generation — f_max = f_min
    # OFF_GRID_TECHS excluded: the current Layers_in_out.csv flags PV_HS/HS_DIESEL as ELECTRICITY
    # generators (positive coefficient) — they weren't in that set when reality/technologies.ipynb
    # last ran (52 generators there vs. 54 here), so without this exclusion Rule (a) would re-lock
    # f_max = f_min = 0 for them here, undoing correction 1 before Rule (b) even runs.
    mask_gen = df["Technologies param"].isin(ELECTRICITY_GENERATORS - set(OFF_GRID_TECHS))
    df.loc[mask_gen, "f_max"] = df.loc[mask_gen, "f_min"]

    # Rule (b): Disable off-grid legacy kits — f_min = 0 only (correction 1: f_max is left
    # unconstrained, inherited from sufficiency, so the model can build new kits)
    for tech in OFF_GRID_TECHS:
        mask = df["Technologies param"] == tech
        df.loc[mask, "f_min"] = 0.0

    # Rule (h): PV_HS / HS_DIESEL f_max_prod capped at the cluster's dispersed demand (see
    # DISPERSED_DEMAND_GWH_BY_CLUSTER above). f_min_prod left at 0 (its default) -- not pinned.
    for tech in OFF_GRID_TECHS:
        mask = df["Technologies param"] == tech
        df.loc[mask, "f_max_prod"] = DISPERSED_DEMAND_GWH_BY_CLUSTER[k]
        df.loc[mask, "f_min_prod"] = 0.0

    # Rule (c): Lock storage — f_max = f_min (no new storage investment); BATT_HS excluded
    # (correction 2, see STORAGE_TECHS definition above)
    mask_stor = df["Technologies param"].isin(STORAGE_TECHS)
    df.loc[mask_stor, "f_max"] = df.loc[mask_stor, "f_min"]

    # Disable ST_SNG in reality scenario: f_min = f_max = 0 for all clusters
    df.loc[df["Technologies param"] == "ST_SNG", "f_min"] = 0.0
    df.loc[df["Technologies param"] == "ST_SNG", "f_max"] = 0.0

    # LED-only lighting: conventional bulb/tube techs disabled (f_max = 0)
    df.loc[df["Technologies param"] == "CONVENTIONAL_BULB",  "f_max"] = 0.0
    df.loc[df["Technologies param"] == "CONVENTIONAL_LIGHT", "f_max"] = 0.0

    # Rule (d): Stove f_min from Census 2024 — f_max unchanged from sufficiency
    # NOTE: not projected to this horizon — see "Missing entries" at the end of this notebook.
    df.loc[df["Technologies param"] == "STOVE_WOOD", "f_min"] = stove_wood_fmin[k]
    df.loc[df["Technologies param"] == "STOVE_LPG",  "f_min"] = stove_lpg_fmin[k]

    capacity_only[k] = df

print("Computed capacity overrides (rules a–e) for C1–C5 — not saved yet, see rule (f) below")

Computed capacity overrides (rules a–e) for C1–C5 — not saved yet, see rule (f) below


## 5. Cost/lifetime block — rule (f)

`c_inv`, `c_maint`, `gwp_constr`, `lifetime` (plus the structural `Category`/`Subcategory`/
`Technologies name`/`Comment` columns) are merged in verbatim from `COST_REF_PATH`
(`Data/2025/reality/02_REF_REGION/Technologies.csv`), the deployed and calibrated Norte Amazónica
2025 catalog — **not** projected or trended in any way for this horizon. Capacity columns
(`c_p`, `fmin_perc`, `fmax_perc`, `f_min`, `f_max`, `f_min_prod`, `f_max_prod`) are exactly the
`capacity_only` dataframes computed in Section 4 (rules a–e) and are untouched by this merge.

A blocking assertion immediately below checks that `c_inv`/`c_maint`/`gwp_constr`/`lifetime` are
identical to the 2025 reference for all 272 technologies in every cluster, with **no exceptions** —
cost trajectories for 2035 are not modelled yet. If the assertion fails, the notebook raises before
writing any `Technologies.csv`, so a partially-wrong catalogue is never deployed.

In [6]:
# Rule (f): merge cost/lifetime block from the calibrated 2025 reality catalog.
# Source: Data/2025/reality/02_REF_REGION/Technologies.csv (Norte Amazónica 2025 calibration).
COST_COLS = ["Category", "Subcategory", "Technologies name", "c_inv", "c_maint", "gwp_constr",
             "lifetime", "Comment"]
DEPLOYED_COLUMNS = ["Category", "Subcategory", "Technologies name", "Technologies param",
                    "c_inv", "c_maint", "gwp_constr", "lifetime",
                    "c_p", "fmin_perc", "fmax_perc", "f_min", "f_max",
                    "f_min_prod", "f_max_prod", "Comment"]

# skiprows=[1]: the deployed file has a units-description row right after the header
cost_ref = pd.read_csv(COST_REF_PATH, sep=";", skiprows=[1])
cost_ref["Technologies param"] = cost_ref["Technologies param"].str.strip()
cost_ref = cost_ref.set_index("Technologies param")

assembled = {}
for k, df in capacity_only.items():
    df = df.set_index("Technologies param")
    missing = set(df.index) - set(cost_ref.index)
    if missing:
        raise ValueError(f"C{k}: technologies missing from the 2025 cost reference: {sorted(missing)}")
    df[COST_COLS] = cost_ref.loc[df.index, COST_COLS]

    # NREL ATB Moderate/Mid trajectory -- ratio applied to c_inv only, never an absolute
    # substitution. c_maint, gwp_constr, lifetime are left untouched (verbatim from cost_ref).
    for tech, ratio_by_year in NREL_ATB_C_INV_RATIO.items():
        if tech in df.index:
            df.loc[tech, "c_inv"] = df.loc[tech, "c_inv"] * ratio_by_year[2035]

    assembled[k] = df.reset_index()[DEPLOYED_COLUMNS]

print(f"Merged cost/lifetime block from {COST_REF_PATH} for C1–C5 "
      f"({len(cost_ref)} technologies)")

Merged cost/lifetime block from ../../../EnergyScope_BO_nord_amazonia/Data/2025/reality/02_REF_REGION/Technologies.csv for C1–C5 (272 technologies)


In [7]:
# Blocking assertion -- c_maint, gwp_constr, lifetime must be IDENTICAL to the 2025 reality
# reference for every technology, with NO exceptions. c_inv must be IDENTICAL to the 2025
# reference for every technology EXCEPT the four NREL ATB technologies (PV_UTILITY, BATT_LI,
# PV_HS, BATT_HS), whose c_inv must instead equal exactly ratio * 2025 c_inv
# (NREL_ATB_C_INV_RATIO, Moderate/Mid trajectory). Raises and halts the notebook before any file
# is written if either check fails.
COST_ASSERT_COLS = ["c_maint", "gwp_constr", "lifetime"]
ref_check = cost_ref[COST_ASSERT_COLS]
ratio_techs = set(NREL_ATB_C_INV_RATIO)

for k, df in assembled.items():
    check = df.set_index("Technologies param")[COST_ASSERT_COLS]
    diff = (check - ref_check.loc[check.index]).abs() > 1e-9
    bad = diff.any(axis=1)
    if bad.any():
        raise AssertionError(
            f"C{k}: cost/lifetime columns differ from the 2025 reality reference for "
            f"{check.index[bad].tolist()} -- no exceptions are allowed at this horizon "
            f"(cost trajectories not yet applied to c_maint/gwp_constr/lifetime)."
        )

    c_inv_check = df.set_index("Technologies param")["c_inv"]
    ref_c_inv = cost_ref["c_inv"]
    flat_techs = [t for t in c_inv_check.index if t not in ratio_techs]
    diff_flat = (c_inv_check.loc[flat_techs] - ref_c_inv.loc[flat_techs]).abs() > 1e-9
    if diff_flat.any():
        raise AssertionError(
            f"C{k}: c_inv differs from the 2025 reality reference for "
            f"{c_inv_check.loc[flat_techs].index[diff_flat].tolist()} -- no exceptions are "
            f"allowed outside NREL_ATB_C_INV_RATIO."
        )
    for tech in ratio_techs:
        actual_ratio = c_inv_check[tech] / ref_c_inv[tech]
        expected_ratio = NREL_ATB_C_INV_RATIO[tech][2035]
        if abs(actual_ratio - expected_ratio) > 1e-9:
            raise AssertionError(
                f"C{k}: {tech} c_inv/2025 ratio = {actual_ratio}, expected exactly "
                f"{expected_ratio} (NREL_ATB_C_INV_RATIO, source: NREL ATB 2024 Moderate/Mid)."
            )

print(f"ASSERT OK -- c_maint, gwp_constr, lifetime identical to 2025 reality for all "
      f"{len(cost_ref)} technologies in C1-C5 (no exceptions); c_inv identical to 2025 reality "
      f"for all but {sorted(ratio_techs)}, which match the announced NREL ATB ratio exactly")


ASSERT OK -- c_maint, gwp_constr, lifetime identical to 2025 reality for all 272 technologies in C1-C5 (no exceptions); c_inv identical to 2025 reality for all but ['BATT_HS', 'BATT_LI', 'PV_HS', 'PV_UTILITY'], which match the announced NREL ATB ratio exactly


In [8]:
for k, df in assembled.items():
    out_path = os.path.join(OUT_DIR, f"C{k}", "Technologies.csv")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    df.to_csv(out_path, sep=";", index=False)

print("Saved Technologies.csv for C1–C5 (full 16-column deployed format)")

Saved Technologies.csv for C1–C5 (full 16-column deployed format)


## 6. Verification

In [9]:
clusters_out = {}
for k in range(1, 6):
    path = os.path.join(OUT_DIR, f"C{k}", "Technologies.csv")
    df = pd.read_csv(path, sep=";")
    df["Technologies param"] = df["Technologies param"].str.strip()
    clusters_out[k] = df

def lookup(df, tech, col):
    row = df.loc[df["Technologies param"] == tech, col]
    return float(row.values[0]) if len(row) else float("nan")

header = f"{'Technology':<32}" + "".join(f"  C{k:>11}" for k in range(1, 6))
sep    = "-" * len(header)

# --- Rule (a)+(e): generators locked, existing fleet matches horizon values ---
print("=== Rule (a)+(e): generators locked (f_max == f_min), existing fleet == horizon values ===")
print(header); print(sep)
for tech in ["GENSET_DIESEL", "PV_UTILITY"]:
    for col in ["f_min", "f_max"]:
        vals = [lookup(clusters_out[k], tech, col) for k in range(1, 6)]
        print(f"{tech+' '+col:<32}" + "".join(f"  {v:>11.5f}" for v in vals))
    locked = ["OK" if abs(lookup(clusters_out[k], tech, "f_min") -
                         lookup(clusters_out[k], tech, "f_max")) < 1e-9
              else "FAIL" for k in range(1, 6)]
    print(f"{'  locked?':<32}" + "".join(f"  {s:>11}" for s in locked))
    for k in range(1, 6):
        expected = EXISTING_FLEET_FMIN_GW[tech].get(k, 0.0)
        actual = lookup(clusters_out[k], tech, "f_min")
        assert abs(actual - expected) < 1e-9, (
            f"{tech} f_min C{k} = {actual}, expected {expected} (EXISTING_FLEET_FMIN_GW)")
        assert abs(lookup(clusters_out[k], tech, "f_max") - actual) < 1e-9, (
            f"{tech} not locked (f_max != f_min) in C{k}")
print("  ASSERT OK — matches EXISTING_FLEET_FMIN_GW and is locked (f_max == f_min) in every cluster")

print()

# --- Rule (b): legacy off-grid kits dead (f_min = 0), but buildable (f_max > 0) ---
print("=== Rule (b): legacy kits dead (f_min = 0 in all clusters), new kits buildable (f_max > 0) ===")
print(header); print(sep)
for tech in ["PV_HS", "HS_DIESEL", "BATT_HS"]:
    for col in ["f_min", "f_max"]:
        vals = [lookup(clusters_out[k], tech, col) for k in range(1, 6)]
        print(f"{tech+' '+col:<32}" + "".join(f"  {v:>11.5f}" for v in vals))
    for k in range(1, 6):
        fmin = lookup(clusters_out[k], tech, "f_min")
        fmax = lookup(clusters_out[k], tech, "f_max")
        assert fmin == 0.0, f"{tech} f_min C{k} = {fmin}, expected 0.0 (legacy kits cancelled)"
        assert fmax > 0.0, f"{tech} f_max C{k} = {fmax}, expected > 0 (model must be able to build new kits)"
print("  ASSERT OK — f_min = 0 (legacy dead) and f_max > 0 (new kits buildable) for PV_HS, HS_DIESEL, BATT_HS in every cluster")

print()

# --- LED-only lighting ---
print("=== LED-only lighting (CONVENTIONAL_BULB / CONVENTIONAL_LIGHT f_max == 0) ===")
print(header); print(sep)
for tech in ["CONVENTIONAL_BULB", "CONVENTIONAL_LIGHT"]:
    vals = [lookup(clusters_out[k], tech, "f_max") for k in range(1, 6)]
    ok   = ["OK" if v == 0.0 else "FAIL" for v in vals]
    print(f"{tech+' f_max':<32}" + "".join(f"  {v:>11.5f}" for v in vals))
    print(f"{'  disabled?':<32}" + "".join(f"  {s:>11}" for s in ok))

print()

# --- Rule (d): stove f_min (frozen at Census 2024 — not projected to this horizon) ---
print("=== Rule (d): stove f_min (Census 2024, NOT projected — see 'Missing entries' note) ===")
print(header); print(sep)
for tech in ["STOVE_WOOD", "STOVE_LPG"]:
    for col in ["f_min", "f_max"]:
        vals = [lookup(clusters_out[k], tech, col) for k in range(1, 6)]
        print(f"{tech+' '+col:<32}" + "".join(f"  {v:>11.5f}" for v in vals))

print()
print("=== Efficiencies frozen: Layers_in_out.csv verified identical to the 2025 reality reference (Section 1) ===")

print()
print("Cooking-demand-vs-stove-capacity check SKIPPED: it requires Demands.csv, which this notebook "
      "deliberately does not generate (Source A/B/C breakdown not yet available for this horizon — "
      "see 'Missing entries' note at the end of this notebook).")

print()

# --- Rule (f): cost/lifetime block spot-check on the four technologies known to have been
# corrupted by the previous (pre-fix) build of this notebook ---
print("=== Rule (f): cost/lifetime spot-check against the 2025 reality reference (C1 shown) ===")
spot_checks = [
    ("GRID",       "c_inv",    0.0),
    ("HVAC_LINE",  "c_inv",    2.0),
    ("HVAC_LINE",  "c_maint",  0.04),
    ("PV_HS",      "c_inv",    2730.0),
    ("PV_HS",      "c_maint",  10.0),
    ("PV_HS",      "lifetime", 20.0),
    ("HS_DIESEL",  "lifetime", 5.0),
    ("BATT_LI",    "lifetime", 15.0),
]
for tech, col, expected in spot_checks:
    actual = lookup(clusters_out[1], tech, col)
    ok = "OK" if abs(actual - expected) < 1e-9 else "FAIL"
    print(f"  {tech:<12} {col:<10} = {actual:>12.5f}  (expected {expected:>10.5f})  [{ok}]")
print()
print("=== Rule (g): DEC_SOLAR.f_max spot-check against the deployed reality_access reference ===")
print(header); print(sep)
vals = [lookup(clusters_out[k], "DEC_SOLAR", "f_max") for k in range(1, 6)]
print(f"{'DEC_SOLAR f_max':<32}" + "".join(f"  {v:>11.5f}" for v in vals))
for k in range(1, 6):
    actual = lookup(clusters_out[k], "DEC_SOLAR", "f_max")
    expected = DEC_SOLAR_FMAX_BY_CLUSTER[k]
    assert abs(actual - expected) < 1e-6, (
        f"DEC_SOLAR f_max C{k} = {actual}, expected {expected} (DEC_SOLAR_FMAX_BY_CLUSTER)")
    assert actual > 0.0, f"DEC_SOLAR f_max C{k} = {actual}, expected > 0 (must not be locked to 0)"
print("  ASSERT OK — DEC_SOLAR.f_max matches the deployed reality_access reference and is nonzero in every cluster")

print()
print("=== Rule (h): PV_HS / HS_DIESEL f_max_prod spot-check against cluster_summary.csv ===")
print(header); print(sep)
for tech in ["PV_HS", "HS_DIESEL"]:
    vals = [lookup(clusters_out[k], tech, "f_max_prod") for k in range(1, 6)]
    print(f"{tech + ' f_max_prod':<32}" + "".join(f"  {v:>11.5f}" for v in vals))
    fminp_vals = [lookup(clusters_out[k], tech, "f_min_prod") for k in range(1, 6)]
    print(f"{tech + ' f_min_prod':<32}" + "".join(f"  {v:>11.5f}" for v in fminp_vals))
    for k in range(1, 6):
        actual = lookup(clusters_out[k], tech, "f_max_prod")
        expected = DISPERSED_DEMAND_GWH_BY_CLUSTER[k]
        assert abs(actual - expected) < 1e-6, (
            f"{tech} f_max_prod C{k} = {actual}, expected {expected} (DISPERSED_DEMAND_GWH_BY_CLUSTER)")
        fminp = lookup(clusters_out[k], tech, "f_min_prod")
        assert fminp == 0.0, f"{tech} f_min_prod C{k} = {fminp}, expected 0.0 (must not be pinned)"
print("  ASSERT OK — PV_HS/HS_DIESEL f_max_prod matches cluster_summary.csv dispersed demand, "
      "f_min_prod stays 0, in every cluster")


=== Rule (a)+(e): generators locked (f_max == f_min), existing fleet == horizon values ===
Technology                        C          1  C          2  C          3  C          4  C          5
------------------------------------------------------------------------------------------------------
GENSET_DIESEL f_min                   0.00000      0.00000      0.05333      0.00742      0.02736
GENSET_DIESEL f_max                   0.00000      0.00000      0.05333      0.00742      0.02736
  locked?                                  OK           OK           OK           OK           OK
PV_UTILITY f_min                      0.00000      0.00000      0.00000      0.00040      0.00510
PV_UTILITY f_max                      0.00000      0.00000      0.00000      0.00040      0.00510
  locked?                                  OK           OK           OK           OK           OK
  ASSERT OK — matches EXISTING_FLEET_FMIN_GW and is locked (f_max == f_min) in every cluster

=== Rule (b): legacy 

STOVE_WOOD f_max                  1000000000000000.00000  1000000000000000.00000  1000000000000000.00000  1000000000000000.00000  1000000000000000.00000


STOVE_LPG f_min                       0.00463      0.00037      0.02649      0.00861      0.01203
STOVE_LPG f_max                   1000000000000000.00000  1000000000000000.00000  1000000000000000.00000  1000000000000000.00000  1000000000000000.00000

=== Efficiencies frozen: Layers_in_out.csv verified identical to the 2025 reality reference (Section 1) ===

Cooking-demand-vs-stove-capacity check SKIPPED: it requires Demands.csv, which this notebook deliberately does not generate (Source A/B/C breakdown not yet available for this horizon — see 'Missing entries' note at the end of this notebook).

=== Rule (f): cost/lifetime spot-check against the 2025 reality reference (C1 shown) ===
  GRID         c_inv      =      0.00000  (expected    0.00000)  [OK]
  HVAC_LINE    c_inv      =      2.00000  (expected    2.00000)  [OK]
  HVAC_LINE    c_maint    =      0.04000  (expected    0.04000)  [OK]
  PV_HS        c_inv      =   1774.50000  (expected 2730.00000)  [FAIL]
  PV_HS        c_maint  

## Missing entries — `Demands.csv` not generated for this horizon

This notebook deliberately does **not** produce `output_energyscope_2035/C{k}/Demands.csv`. Building
it the way `demande.ipynb` does for 2025 needs three horizon-specific inputs that don't exist yet:

1. **Source A (grid-connected demand)** — a 2035 equivalent of
   `exctraction of data/output/source_A_all_sectors_end_uses.csv`: AETN grid consumption by
   municipality/sector/end-use, projected to 2035. Only the 2024/2025 measured file exists today.
2. **Source B (off-grid RAMP demand)** — a 2035 equivalent of `data ramp/ramp_reality_annual_summary.csv`:
   a RAMP simulation run for the *projected* number of off-grid households per municipality at 2035
   (not the 2025 figure of 9,325 HH). The projected off-grid (dispersed) household count itself is
   already available at cluster level from the breakeven classification in
   `analyse_GIS_phase2_projections/output/2035/community_detail.csv`, but no RAMP run has been done
   yet against that projected household count, and no per-municipality (rather than per-cluster)
   breakdown exists.
3. **Cooking fuel mix** — a 2035 equivalent of `exctraction of data/output/CSV_final_in_excel.xlsx`:
   projected non-electric cooking households (wood/LPG) by municipality. Only the 2024 census split
   exists today; `STOVE_WOOD`/`STOVE_LPG` `f_min` in this notebook's `Technologies.csv` output is
   still computed from the unprojected 2024 census figures (Section 3 / rule (d)) — an implicit
   "frozen cooking behaviour" assumption, not requested for this horizon and not yet corrected.

`home_systems.ipynb` and `demande.ipynb` were copied into this folder (with their output path
suffixed to `output_energyscope_2035`, per correction 4) but were **not executed**, for the same
reason: their 2025 inputs (2024 census equipment counts, `source_A_all_sectors_end_uses.csv`,
`ramp_reality_annual_summary.csv`) are not valid for 2035, and no projected replacement exists yet.
`home_systems.ipynb`'s output (legacy off-grid kit `f_min`) is moot regardless of that gap — those
values are forced to 0 directly in this notebook (rule (b) / content rule "kits legacy annulés"),
since the 2012-vintage kits are past their catalogue lifetime at this horizon.

## 7. Deploy `Layers_in_out.csv` to the case-study data directories

The project rule is frozen efficiencies: `Layers_in_out.csv` must be byte-identical to the 2025
reality reference (`{LIO_REFERENCE_PATH}`) at every horizon. Section 1 above already verifies the
local source file (`../data/Layers_in_out.csv`) matches it.

This closes a second, separate gap found while attempting the first 2035 case-study rerun: the
deployed `Data/2035/*/00_INDEP/Layers_in_out.csv` copies used at solve time had drifted from that
local source and picked up an extra `DEC_ELEC_COLD_FAN` row absent from the 2025 reference. That
row is not referenced by any `Demands.csv` (2025 or 2035 projections) or by any `.mod` set, so
dropping it is safe. Each deployed copy below is overwritten with an exact copy of the 2025
reference and re-verified byte-for-byte plus value-for-value (`DataFrame.equals`) — no partial or
edited copies.


In [10]:
# Section 7: overwrite the deployed Layers_in_out.csv copies with an exact copy of the
# 2025 reality reference (frozen-efficiencies rule). Byte-exact copy + re-verification, no manual
# editing of any deployed CSV.
import filecmp
import shutil

DEPLOY_SCENARIOS_2035 = ["no_transition", "early_access", "late_access"]
DEPLOY_TARGETS_2035 = [
    f"../../../EnergyScope_BO_nord_amazonia/Data/2035/{scenario}/00_INDEP/Layers_in_out.csv"
    for scenario in DEPLOY_SCENARIOS_2035
]

lio_reference_check = pd.read_csv(LIO_REFERENCE_PATH, sep=";", header=0, index_col=0)

for target in DEPLOY_TARGETS_2035:
    shutil.copyfile(LIO_REFERENCE_PATH, target)

    # byte-exact check
    if not filecmp.cmp(LIO_REFERENCE_PATH, target, shallow=False):
        raise AssertionError(f"{target} is not byte-identical to the 2025 reality reference after deployment")

    # value-exact check (same pattern as the Section 1 assertion)
    deployed_check = pd.read_csv(target, sep=";", header=0, index_col=0)
    if not deployed_check.equals(lio_reference_check):
        raise AssertionError(f"{target} does not match the 2025 reality reference after deployment")

    print(f"OK — {target} deployed and verified byte-identical to the 2025 reality reference")


OK — ../../../EnergyScope_BO_nord_amazonia/Data/2035/no_transition/00_INDEP/Layers_in_out.csv deployed and verified byte-identical to the 2025 reality reference


OK — ../../../EnergyScope_BO_nord_amazonia/Data/2035/early_access/00_INDEP/Layers_in_out.csv deployed and verified byte-identical to the 2025 reality reference
OK — ../../../EnergyScope_BO_nord_amazonia/Data/2035/late_access/00_INDEP/Layers_in_out.csv deployed and verified byte-identical to the 2025 reality reference


## 8. Deploy `Technologies.csv` (capacity columns) to the case-study data directories

`Data/2035/*/C{k}/Technologies.csv` (the minimal 8-column capacity file actually read by
`esmc.utils.region.py:read_tech()` and merged onto the `02_REF_REGION` catalog at solve time) was
deployed from an earlier build of this notebook's `output_energyscope_2035/C{k}/Technologies.csv`
and has gone stale on `DEC_SOLAR.f_max` (Rule (g) above) since. Redeploy the capacity columns for
all 5 clusters, in every 2035 scenario, from the just-recomputed `assembled` dataframes — no
manual CSV edits. Re-verified against the deployed `DEC_SOLAR_FMAX_BY_CLUSTER` reference below.


In [11]:
# Section 8: redeploy the 8-column capacity subset of Technologies.csv to every 2035
# case-study cluster directory, from the just-recomputed `assembled` dataframes (Section 5). Only
# the capacity columns are deployed here -- the cost/lifetime block in 02_REF_REGION is untouched
# (already correctly deployed, unaffected by the DEC_SOLAR fix).
CAPACITY_COLS = ["Technologies param", "c_p", "fmin_perc", "fmax_perc",
                  "f_min", "f_max", "f_min_prod", "f_max_prod"]

for scenario in DEPLOY_SCENARIOS_2035:
    for k in range(1, 6):
        target = (f"../../../EnergyScope_BO_nord_amazonia/Data/2035/{scenario}/"
                  f"C{k}/Technologies.csv")
        assembled[k][CAPACITY_COLS].to_csv(target, sep=";", index=False)

        redeployed = pd.read_csv(target, sep=";", index_col=0)
        redeployed.index = redeployed.index.str.strip()
        actual = float(redeployed.loc["DEC_SOLAR", "f_max"])
        expected = DEC_SOLAR_FMAX_BY_CLUSTER[k]
        if abs(actual - expected) > 1e-6 or actual <= 0.0:
            raise AssertionError(
                f"{target}: DEC_SOLAR f_max = {actual}, expected {expected} (nonzero)")
        for tech in ["PV_HS", "HS_DIESEL"]:
            actual_fmp = float(redeployed.loc[tech, "f_max_prod"])
            expected_fmp = DISPERSED_DEMAND_GWH_BY_CLUSTER[k]
            if abs(actual_fmp - expected_fmp) > 1e-6:
                raise AssertionError(
                    f"{target}: {tech} f_max_prod = {actual_fmp}, expected {expected_fmp}")
            actual_fminp = float(redeployed.loc[tech, "f_min_prod"])
            if actual_fminp != 0.0:
                raise AssertionError(f"{target}: {tech} f_min_prod = {actual_fminp}, expected 0.0")
    print(f"OK — Technologies.csv (C1-C5) redeployed for 2035/{scenario}, "
          f"DEC_SOLAR.f_max verified nonzero and matching reality_access in every cluster")


OK — Technologies.csv (C1-C5) redeployed for 2035/no_transition, DEC_SOLAR.f_max verified nonzero and matching reality_access in every cluster


OK — Technologies.csv (C1-C5) redeployed for 2035/early_access, DEC_SOLAR.f_max verified nonzero and matching reality_access in every cluster


OK — Technologies.csv (C1-C5) redeployed for 2035/late_access, DEC_SOLAR.f_max verified nonzero and matching reality_access in every cluster


## 8bis. Deploy cost/lifetime block (`c_inv` only changes) to `02_REF_REGION`

No prior step wrote the cost/lifetime block to `Data/2035/{scenario}/02_REF_REGION/Technologies.csv` -- it has so far been a static file, correctly matching the 2025 reality reference by construction (Section 5), never rewritten because nothing needed updating. Now that the NREL ATB ratio makes `c_inv` diverge from 2025 for four technologies, this step becomes necessary. Only the four cost/lifetime columns are touched; every other column (capacity, category, comment placeholder text) in the deployed file is preserved untouched, and the file's header + units row are preserved byte-for-byte.

In [12]:
COST_ONLY_COLS = ["c_inv", "c_maint", "gwp_constr", "lifetime"]

for scenario in DEPLOY_SCENARIOS_2035:
    target = (f"../../../EnergyScope_BO_nord_amazonia/Data/2035/{scenario}/"
              f"02_REF_REGION/Technologies.csv")

    with open(target, encoding="utf-8") as f:
        header_line = f.readline()
        units_line = f.readline()

    existing = pd.read_csv(target, sep=";", skiprows=[1])
    existing["Technologies param"] = existing["Technologies param"].str.strip()
    existing = existing.set_index("Technologies param")

    missing = set(cost_ref.index) - set(existing.index)
    if missing:
        raise ValueError(
            f"{target}: technologies missing from the deployed ref-region catalog: "
            f"{sorted(missing)}")

    # cost columns are cluster-independent (identical across C1-C5) -- any k works as source
    source_cost = assembled[1].set_index("Technologies param")[COST_ONLY_COLS]
    existing.loc[source_cost.index, COST_ONLY_COLS] = source_cost

    for tech in sorted(ratio_techs):
        ratio = NREL_ATB_C_INV_RATIO[tech][2035]
        note = (f"NREL ATB 2024 Moderate/Mid c_inv ratio 2035={ratio} "
                f"(source: analyse data ramp/2035/technologies.ipynb).")
        prior = existing.loc[tech, "Comment"]
        prior = "" if pd.isna(prior) else str(prior).strip()
        existing.loc[tech, "Comment"] = (prior + " " + note).strip() if prior else note

    with open(target, "w", encoding="utf-8", newline="") as f:
        f.write(header_line)
        f.write(units_line)
        existing.reset_index()[DEPLOYED_COLUMNS].to_csv(f, sep=";", index=False, header=False)

    # re-read and verify
    redeployed = pd.read_csv(target, sep=";", skiprows=[1])
    redeployed["Technologies param"] = redeployed["Technologies param"].str.strip()
    redeployed = redeployed.set_index("Technologies param")
    for tech in sorted(ratio_techs):
        actual_ratio = redeployed.loc[tech, "c_inv"] / cost_ref.loc[tech, "c_inv"]
        expected_ratio = NREL_ATB_C_INV_RATIO[tech][2035]
        if abs(actual_ratio - expected_ratio) > 1e-9:
            raise AssertionError(
                f"{target}: deployed {tech} c_inv/2025 ratio = {actual_ratio}, expected "
                f"exactly {expected_ratio}")
    other_techs = [t for t in redeployed.index if t not in ratio_techs]
    diff_other = (redeployed.loc[other_techs, "c_inv"] - cost_ref.loc[other_techs, "c_inv"]).abs() > 1e-9
    if diff_other.any():
        raise AssertionError(
            f"{target}: c_inv changed for non-ATB technologies: "
            f"{redeployed.loc[other_techs].index[diff_other].tolist()}")

    print(f"OK -- {target} cost/lifetime block redeployed, c_inv ratio verified for "
          f"{sorted(ratio_techs)}, unchanged for every other technology")


OK -- ../../../EnergyScope_BO_nord_amazonia/Data/2035/no_transition/02_REF_REGION/Technologies.csv cost/lifetime block redeployed, c_inv ratio verified for ['BATT_HS', 'BATT_LI', 'PV_HS', 'PV_UTILITY'], unchanged for every other technology
OK -- ../../../EnergyScope_BO_nord_amazonia/Data/2035/early_access/02_REF_REGION/Technologies.csv cost/lifetime block redeployed, c_inv ratio verified for ['BATT_HS', 'BATT_LI', 'PV_HS', 'PV_UTILITY'], unchanged for every other technology
OK -- ../../../EnergyScope_BO_nord_amazonia/Data/2035/late_access/02_REF_REGION/Technologies.csv cost/lifetime block redeployed, c_inv ratio verified for ['BATT_HS', 'BATT_LI', 'PV_HS', 'PV_UTILITY'], unchanged for every other technology


C:\Users\valen\AppData\Local\Temp\ipykernel_10280\192933574.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'NREL ATB 2024 Moderate/Mid c_inv ratio 2035=0.76 (source: analyse data ramp/2035/technologies.ipynb).' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  existing.loc[tech, "Comment"] = (prior + " " + note).strip() if prior else note
C:\Users\valen\AppData\Local\Temp\ipykernel_10280\192933574.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'NREL ATB 2024 Moderate/Mid c_inv ratio 2035=0.76 (source: analyse data ramp/2035/technologies.ipynb).' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  existing.loc[tech, "Comment"] = (prior + " " + note).strip() if prior else note
C:\Users\valen\AppData\Local\Temp\ipykernel_10280\192933574.

## 9. Pass 2 — calibrate `share_dispersion` from pass-1 output (2035 only)

Pass 1 (`share_dispersion=0` everywhere) validated the `f_max_prod` cap on `PV_HS`/`HS_DIESEL`
(Rule (h)) but left dispersed households served by the grid instead of home systems: `TECH_HS`
came out at zero in all 5 clusters of `early_access`/`late_access`, and in C1 of `no_transition`,
even though dispersed demand there is nonzero.

`share_tech_hs` (`ESMC_model_AMPL.mod:642-644`) is an **hourly** share constraint: `TECH_HS`'s net
electricity output must be at least `share_dispersion[c]` times total non-storage electricity
production+resources (its own `TECH_HS diff STORAGE_TECH` output included in that total, along with
`R_t_local`/`R_t_exterior`, excluding inter-cluster `R_t_import`/`R_t_export`). Calibrating
`share_dispersion[c]` from pass-1's own output is a fixed-point approximation: pass 1 had
`TECH_HS ≈ 0`, so pass 1's "everything else" is a good annual estimate of the non-`TECH_HS` supply
pass 2 needs to divide against --

```
share_dispersion[c] = target[c] / (target[c] + offre_locale_hors_TECH_HS[c])
```

where `offre_locale_hors_TECH_HS[c]` is computed per the calibrated pass-1 case-study output:
production of every technology feeding `ELECTRICITY` (excluding `PV_HS`, `HS_DIESEL`, `BATT_HS`,
matching the `.mod`'s `diff STORAGE_TECH` on the `TECH_HS`-storage side) plus `R_year_local` +
`R_year_exterior` from that cluster's own `Resources.csv` -- **excluding** `R_year_import`/
`R_year_export` (inter-cluster exchange; `Year_balance.csv`'s `ELECTRICITY` row nets these together
with exterior import, so `Resources.csv` is read directly instead, per-region). `C5` stays 0
(zero dispersed demand there in every scenario).

**C2 special case:** C2 imports nearly all its electricity from C5 (near-zero non-`TECH_HS` local
supply in `no_transition`'s pass 1, small in `early_access`/`late_access`) -- the denominator is
too close to zero for any `share_dispersion` value to reliably bite. `share_dispersion[C2]` is
still computed and deployed (descriptive), but the actual mechanism used there is `PV_HS.f_min_prod
= 0.130` (C2's target) with `PV_HS.f_max_prod` left open (`1e15`) in C2 only, to avoid pinning
(`f_min_prod == f_max_prod` breaks the solver). `HS_DIESEL.f_max_prod` in C2 is untouched (stays at
Rule (h)'s 0.130). The already-deployed `f_max_prod` caps elsewhere are not touched.


---

**Pass 3 correction:** two problems found after reading pass-2 results.

1. `share_dispersion[C2]` was deployed as an *active* model parameter, but was only ever meant as a
   descriptive quantity for the writeup -- the C2 fallback (`PV_HS.f_min_prod` + open `f_max_prod`)
   was supposed to be the only live mechanism there. With `share_dispersion[C2]` nonzero in the
   `.dat` and `PV_HS.f_max_prod` open, the two interacted in `no_transition` (where `PV_UTILITY` is
   locked and `GENSET_DIESEL` dominates, leaving almost no non-`TECH_HS` local supply) and built
   1.569 GWh of `PV_HS`+`HS_DIESEL` against a 0.130 GWh target -- a ~12x overshoot. Fixed:
   `share_dispersion[C2]` is now forced to 0 at deploy time in all three scenarios (computed value
   still printed for the record, just not written into the `.dat`), and C2's `PV_HS` reverts to the
   uniform Rule (h) regime (`f_min_prod=0`, `f_max_prod=0.130`, same as every other cluster and same
   as `HS_DIESEL` in C2) -- no floor, no open ceiling, no special case.
2. `share_dispersion[c]` for C1/C3/C4 is recalibrated from **pass 2's own output** (same formula,
   same denominator definition, same 2035 targets) rather than pass 1's -- a second fixed-point
   iteration, now that pass 2 shows what the system looks like with `TECH_HS` actually contributing.


In [12]:
# Section 9a (pass 3): compute share_dispersion[c] per scenario from that scenario's own
# PASS-2 output (same case-study paths -- pass 2 overwrote pass 1's output in place).
# Same formula/denominator as before, second fixed-point iteration.
PASS1_CS_ROOT = "../../../EnergyScope_BO_nord_amazonia/case_studies/C1_C2_C3_C4_C5"
EXCLUDE_TECHS_FOR_OFFRE_LOCALE = {"ELECTRICITY", "PV_HS", "HS_DIESEL", "BATT_HS"}

SHARE_DISPERSION_BY_SCENARIO = {}  # scenario -> {k: share_dispersion}
OFFRE_LOCALE_BY_SCENARIO = {}      # scenario -> {k: offre_locale_hors_TECH_HS}, kept for the record

for scenario in DEPLOY_SCENARIOS_2035:
    cs_dir = f"{PASS1_CS_ROOT}/norte_amazonia_{scenario}_2035/outputs/regional_results"

    yb = pd.read_csv(f"{cs_dir}/Year_balance.csv", sep=";")
    yb["ELECTRICITY"] = pd.to_numeric(yb["ELECTRICITY"], errors="coerce")
    yb_tech = yb[~yb["Elements"].isin(EXCLUDE_TECHS_FOR_OFFRE_LOCALE)]
    yb_tech = yb_tech[yb_tech["ELECTRICITY"] > 1e-9]
    tech_sum = yb_tech.groupby("Regions")["ELECTRICITY"].sum()

    res = pd.read_csv(f"{cs_dir}/Resources.csv", sep=";")
    res_elec = res[res["Resources"] == "ELECTRICITY"].set_index("Regions")
    r_local = pd.to_numeric(res_elec["R_year_local"], errors="coerce").fillna(0.0)
    r_ext = pd.to_numeric(res_elec["R_year_exterior"], errors="coerce").fillna(0.0)
    # R_year_import / R_year_export deliberately excluded -- inter-cluster exchange, not local supply

    shares, offres = {}, {}
    for k in range(1, 6):
        region = f"C{k}"
        offre_locale = (float(tech_sum.get(region, 0.0))
                         + float(r_local.get(region, 0.0)) + float(r_ext.get(region, 0.0)))
        target = DISPERSED_DEMAND_GWH_BY_CLUSTER[k]
        if k == 5:
            share = 0.0
        elif (target + offre_locale) > 0:
            share = target / (target + offre_locale)
        else:
            share = 0.0
        shares[k] = share
        offres[k] = offre_locale
    SHARE_DISPERSION_BY_SCENARIO[scenario] = shares
    OFFRE_LOCALE_BY_SCENARIO[scenario] = offres
    print(f"{scenario}_2035: offre_locale_hors_TECH_HS = {offres}")
    print(f"{scenario}_2035: share_dispersion         = {shares}")


no_transition_2035: offre_locale_hors_TECH_HS = {1: 24.206437645469013, 2: 0.0, 3: 130.97783455222233, 4: 52.59888339807611, 5: 68.34925573348904}
no_transition_2035: share_dispersion         = {1: 0.07690184589840762, 2: 1.0, 3: 0.005387761871044922, 4: 0.02814198416930415, 5: 0.0}
early_access_2035: offre_locale_hors_TECH_HS = {1: 31.71231279099324, 2: 3.0576492154704837, 3: 175.87433356143217, 4: 66.56431546897618, 5: 90.71920371382411}
early_access_2035: share_dispersion         = {1: 0.05978846731574758, 2: 0.04081249533953642, 3: 0.004017921605225376, 4: 0.022369772585860537, 5: 0.0}
late_access_2035: offre_locale_hors_TECH_HS = {1: 24.206812906429445, 2: 2.292444358109452, 3: 160.2537989226605, 4: 51.008086680971516, 5: 89.74779436565021}
late_access_2035: share_dispersion         = {1: 0.07690074542149206, 2: 0.05370386699607421, 3: 0.004407837095466712, 4: 0.028994205085256824, 5: 0.0}


In [13]:
# Section 9b (pass 3): deploy share_dispersion into each scenario's Misc.json (C1-C5), with
# assertion. C2 is forced to 0 at deploy time -- share_dispersion[C2] is descriptive only (for the
# writeup), never an active model parameter; the computed value is still printed above for the
# record, just not written into the .dat. See pass-3 correction note in Section 9 markdown.
import json as _json

for scenario in DEPLOY_SCENARIOS_2035:
    for k in range(1, 6):
        target = f"../../../EnergyScope_BO_nord_amazonia/Data/2035/{scenario}/C{k}/Misc.json"
        deployed_value = 0.0 if k == 2 else SHARE_DISPERSION_BY_SCENARIO[scenario][k]

        with open(target, encoding="utf-8") as f:
            misc = _json.load(f)
        misc["share_dispersion"] = deployed_value
        with open(target, "w", encoding="utf-8") as f:
            _json.dump(misc, f, indent=4)

        with open(target, encoding="utf-8") as f:
            check = _json.load(f)
        actual = check["share_dispersion"]
        if abs(actual - deployed_value) > 1e-9:
            raise AssertionError(f"{target}: share_dispersion = {actual}, expected {deployed_value}")
    print(f"OK — share_dispersion deployed and verified in Misc.json (C1-C5) for 2035/{scenario} "
          f"(C2 forced to 0, descriptive value was {SHARE_DISPERSION_BY_SCENARIO[scenario][2]:.6f})")


OK — share_dispersion deployed and verified in Misc.json (C1-C5) for 2035/no_transition (C2 forced to 0, descriptive value was 1.000000)


OK — share_dispersion deployed and verified in Misc.json (C1-C5) for 2035/early_access (C2 forced to 0, descriptive value was 0.040812)
OK — share_dispersion deployed and verified in Misc.json (C1-C5) for 2035/late_access (C2 forced to 0, descriptive value was 0.053704)


In [14]:
# Section 9c (pass 3 correction): C2 special case REMOVED. PV_HS in C2 reverts to the uniform
# Rule (h) regime -- f_min_prod=0 (no floor), f_max_prod=target[C2]=0.130 (cap only), identical to
# every other cluster and to HS_DIESEL in C2. The pass-2 fallback (f_min_prod floor + open
# f_max_prod ceiling) is gone. Explicitly (re)written and asserted rather than left implicit, even
# though Section 8's redeploy already writes this uniform value from Rule (h) directly (Rule (h)
# never special-cased C2 -- only this section used to, and no longer does).
C2_TARGET = DISPERSED_DEMAND_GWH_BY_CLUSTER[2]

for scenario in DEPLOY_SCENARIOS_2035:
    target = f"../../../EnergyScope_BO_nord_amazonia/Data/2035/{scenario}/C2/Technologies.csv"
    df = pd.read_csv(target, sep=";", index_col=0)
    df.index = df.index.str.strip()
    df.loc["PV_HS", "f_min_prod"] = 0.0
    df.loc["PV_HS", "f_max_prod"] = C2_TARGET
    df.to_csv(target, sep=";")

    check = pd.read_csv(target, sep=";", index_col=0)
    check.index = check.index.str.strip()
    if float(check.loc["PV_HS", "f_min_prod"]) != 0.0:
        raise AssertionError(f"{target}: PV_HS f_min_prod = {check.loc['PV_HS', 'f_min_prod']}, expected 0.0")
    if abs(float(check.loc["PV_HS", "f_max_prod"]) - C2_TARGET) > 1e-9:
        raise AssertionError(f"{target}: PV_HS f_max_prod = {check.loc['PV_HS', 'f_max_prod']}, expected {C2_TARGET}")
    if abs(float(check.loc["HS_DIESEL", "f_max_prod"]) - C2_TARGET) > 1e-9:
        raise AssertionError(
            f"{target}: HS_DIESEL f_max_prod = {check.loc['HS_DIESEL', 'f_max_prod']}, "
            f"expected unchanged at {C2_TARGET}")
    print(f"OK — 2035/{scenario} C2 reverted to uniform regime: PV_HS f_min_prod=0.0, "
          f"f_max_prod={C2_TARGET} (cap only, same as HS_DIESEL and every other cluster)")


OK — 2035/no_transition C2 reverted to uniform regime: PV_HS f_min_prod=0.0, f_max_prod=0.1301 (cap only, same as HS_DIESEL and every other cluster)
OK — 2035/early_access C2 reverted to uniform regime: PV_HS f_min_prod=0.0, f_max_prod=0.1301 (cap only, same as HS_DIESEL and every other cluster)
OK — 2035/late_access C2 reverted to uniform regime: PV_HS f_min_prod=0.0, f_max_prod=0.1301 (cap only, same as HS_DIESEL and every other cluster)
